In [44]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

params_file = (
    project_root
    / "models"
    / "best_gradient_boosting_params.json"
)

with open(params_file, "r") as f:
    best_params = json.load(f)

print("Dataset shape:", df.shape)
print()
print(df["split"].value_counts())

print()
print("Locked model parameters:")
print(best_params)

Dataset shape: (12515, 24)

split
train         8789
test          1866
validation    1860
Name: count, dtype: int64

Locked model parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__min_samples_leaf': 3, 'model__n_estimators': 200}


In [45]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

target = "future_spo2_5min"

print("Final number of features:", len(spo2_hr_features))

Final number of features: 11


In [46]:
development_df = df[
    df["split"].isin(
        ["train", "validation"]
    )
].copy()

test_df = df[
    df["split"] == "test"
].copy()

development_ready = development_df.dropna(
    subset=[target]
).copy()

test_eval = test_df.dropna(
    subset=[
        "spo2",
        target
    ]
).copy()

print(
    "Development patients:",
    development_ready["caseid"].nunique()
)

print(
    "Test patients:",
    test_eval["caseid"].nunique()
)

print(
    "Development rows:",
    len(development_ready)
)

print(
    "Test evaluation rows:",
    len(test_eval)
)

Development patients: 42
Test patients: 8
Development rows: 10072
Test evaluation rows: 1750


In [47]:
development_ids = set(
    development_ready["caseid"].unique()
)

test_ids = set(
    test_eval["caseid"].unique()
)

overlap = development_ids & test_ids

print(
    "Patient overlap:",
    overlap
)

Patient overlap: set()


In [48]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

final_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

final_model.set_params(
    **best_params
)

print(final_model)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, max_depth=2,
                                           min_samples_leaf=3, n_estimators=200,
                                           random_state=42))])


In [49]:
X_development = development_ready[
    spo2_hr_features
]

y_development = development_ready[
    target
]

final_model.fit(
    X_development,
    y_development
)

print("Final model training complete!")

Final model training complete!


In [50]:
X_test = test_eval[
    spo2_hr_features
]

y_test = test_eval[
    target
]

y_test_pred = final_model.predict(
    X_test
)

print(
    "Test predictions:",
    len(y_test_pred)
)

Test predictions: 1750


In [51]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("=== FINAL TEST RESULTS ===")
print()

print(
    f"MAE:  {final_mae:.6f}"
)

print(
    f"RMSE: {final_rmse:.6f}"
)

print(
    f"R²:   {final_r2:.6f}"
)

=== FINAL TEST RESULTS ===

MAE:  0.537516
RMSE: 1.181831
R²:   0.297616


In [52]:
persistence_pred = test_eval["spo2"]

persistence_mae = mean_absolute_error(
    y_test,
    persistence_pred
)

persistence_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        persistence_pred
    )
)

persistence_r2 = r2_score(
    y_test,
    persistence_pred
)

print("=== TEST PERSISTENCE BASELINE ===")
print()

print(
    f"MAE:  {persistence_mae:.6f}"
)

print(
    f"RMSE: {persistence_rmse:.6f}"
)

print(
    f"R²:   {persistence_r2:.6f}"
)

=== TEST PERSISTENCE BASELINE ===

MAE:  0.374571
RMSE: 1.212966
R²:   0.260121


In [53]:
final_results = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Tuned Gradient Boosting + HR"
    ],

    "MAE": [
        persistence_mae,
        final_mae
    ],

    "RMSE": [
        persistence_rmse,
        final_rmse
    ],

    "R2": [
        persistence_r2,
        final_r2
    ]
})

final_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.374571,1.212966,0.260121
1,Tuned Gradient Boosting + HR,0.537516,1.181831,0.297616


In [54]:
final_results_file = (
    project_root
    / "data"
    / "processed"
    / "final_test_results.csv"
)

final_results.to_csv(
    final_results_file,
    index=False
)

print(
    "Final test results saved!"
)

Final test results saved!


In [55]:
final_results_file = (
    project_root
    / "data"
    / "processed"
    / "final_test_results.csv"
)

final_results.to_csv(
    final_results_file,
    index=False
)

print("Final test results saved!")

Final test results saved!


In [56]:
test_output = test_eval[
    [
        "caseid",
        "minute",
        "spo2",
        "future_spo2_5min"
    ]
].copy()

test_output["persistence_prediction"] = persistence_pred.values
test_output["ml_prediction"] = y_test_pred

test_output["persistence_error"] = (
    test_output["future_spo2_5min"]
    - test_output["persistence_prediction"]
).abs()

test_output["ml_error"] = (
    test_output["future_spo2_5min"]
    - test_output["ml_prediction"]
).abs()

prediction_file = (
    project_root
    / "data"
    / "processed"
    / "final_test_predictions.csv"
)

test_output.to_csv(
    prediction_file,
    index=False
)

print("Final test predictions saved!")

Final test predictions saved!


In [57]:
import joblib

model_file = (
    project_root
    / "models"
    / "final_gradient_boosting_model.joblib"
)

joblib.dump(
    final_model,
    model_file
)

print("Final model saved to:")
print(model_file)

Final model saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\models\final_gradient_boosting_model.joblib


In [58]:
loaded_model = joblib.load(model_file)

sample_prediction = loaded_model.predict(
    X_test.iloc[[0]]
)

print("Loaded model prediction:", sample_prediction)

Loaded model prediction: [98.47687786]


In [59]:
import json

final_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5",
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

features_file = (
    project_root
    / "models"
    / "final_features.json"
)

with open(features_file, "w") as f:
    json.dump(
        final_features,
        f,
        indent=4
    )

print("Final features saved to:")
print(features_file)

Final features saved to:
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\models\final_features.json
